# 🏛️ Microsoft Fabric Data Plan Architectural Assessment
### *Essential Pre-Implementation Discovery Checklist based on Fundamentals of Data Engineering (Reis & Housley)*

> **Target Platform:** Microsoft Fabric (OneLake, Lakehouse, Data Factory Pipelines, Synapse Spark, Direct Lake Power BI)  
> **Methodology:** The Data Engineering Lifecycle (*Fundamentals of Data Engineering*, Chapter 2)  
> **Purpose:** Evaluate architectural feasibility, operational risks, scalability, and business ROI **before** deploying data pipelines or provisioning compute in Microsoft Fabric.

---

## 🧭 The Data Engineering Lifecycle Framework
According to Joe Reis and Matt Housley (*Fundamentals of Data Engineering*, O'Reilly), data engineering revolves around a core 5-stage lifecycle supported by critical undercurrents:

```
[ 1. Generation ] ───▶ [ 2. Storage ] ───▶ [ 3. Ingestion ] ───▶ [ 4. Transformation ] ───▶ [ 5. Serving (BI/ML/App) ]
  (Source Systems)        (OneLake / Delta)      (Pipelines / Streams)     (Notebooks / Spark / dbt)       (Direct Lake / MLflow)
──────────────────────────────────────────────────────────────────────────────────────────────────────────
                       CROSS-CUTTING UNDERCURRENTS:
  • Security & Privacy   • Data Governance & Lineage (Purview)   • DataOps & CI/CD   • Architecture & Capacity (CU)
```

This notebook acts as an **interactive readiness assessment**. Work through each section with your architecture, engineering, and business stakeholders before committing to pipeline implementation in Microsoft Fabric.


---
## 1. Generation / Source Systems (Book Reference: Page 36)
> *"Before even ingesting data, a data engineer must evaluate how the source systems generate data by answering these big questions about source systems."* — Reis & Housley

### 📋 Architectural Questions & Microsoft Fabric Context

1. **Schema Evolution Policy:** *If schema changes (say, a new column is added), how is this dealt with and communicated to downstream stakeholders?*
   * **Fabric Context:** Establish explicit **Data Contracts**. For Lakehouse Delta tables, decide whether to allow schema drift via Delta Lake Schema Evolution (`mergeSchema = true`) or enforce strict schemas to quarantine unexpected columns.
2. **Pull Frequency:** *How frequently should data be pulled from the source system?*
   * **Fabric Context:** Align extraction frequency with Fabric Capacity Units (CU) and business latency requirements. Distinguish batch pipelines (Fabric Data Factory) from continuous micro-batches (Fabric Eventstreams / Real-Time Hub).
3. **Stateful Systems & CDC:** *For stateful systems (e.g., a database tracking customer account information), is data provided as periodic snapshots or update events from change data capture (CDC)? What’s the logic for how changes are performed, and how are these tracked in the source database?*
   * **Fabric Context:** Use **Fabric Mirrored Databases** (Azure SQL, Cosmos DB, Snowflake) for automated zero-ETL CDC replication into OneLake Delta tables, avoiding expensive custom diff extraction pipelines.
4. **Data Provider Transmission:** *Who/what is the data provider that will transmit the data for downstream consumption?*
   * **Fabric Context:** Clarify whether the pattern is **Source-Push** (Webhooks, Azure Event Hubs) or **Fabric-Pull** (Data Factory Copy Activity via On-Premises Data Gateway or Managed VNet).
5. **Source System Impact:** *Will reading from a data source impact its performance?*
   * **Fabric Context:** Point pipeline connections to read-only database replicas or execute queries during off-peak windows to prevent row-level lock contention on production OLTP workloads.
6. **Upstream Dependencies:** *Does the source system have upstream data dependencies? What are the characteristics of these upstream systems?*
   * **Fabric Context:** Build dependency-aware pipeline triggers in Fabric Data Factory (e.g., waiting for validation flags or file watermarks) rather than brittle, hardcoded time schedules.
7. **Data Quality Checks (Late / Missing Data):** *Are data-quality checks in place to check for late or missing data?*
   * **Fabric Context:** Implement watermark assertions and arrival threshold checks in Bronze-to-Silver PySpark notebooks. Connect validation metrics to **Fabric Data Activator (Reflex)** for automated alerting.


In [ ]:
# ==============================================================================
# 🛠️ STAGE 1: SOURCE SYSTEM DISCOVERY MATRIX
# ==============================================================================
import pandas as pd

source_assessment = [
    {
        "ID": "1.1",
        "Category": "Generation",
        "Question": "Schema Evolution Policy & Communication",
        "Status": "Action Required", # Verified | In Progress | Action Required | Not Applicable
        "Answer_Details": "Source system is MySQL OLTP; schema migrations happen bi-weekly without advance notification.",
        "Risk_Level": "HIGH",        # LOW | MEDIUM | HIGH | CRITICAL
        "Fabric_Mitigation": "Implement Delta Lake mergeSchema in Bronze layer and configure schema drift alerts in Fabric Pipeline."
    },
    {
        "ID": "1.2",
        "Category": "Generation",
        "Question": "Pull Frequency & Cadence",
        "Status": "Verified",
        "Answer_Details": "Daily batch extraction at 02:00 UTC meets business SLA of 08:00 UTC.",
        "Risk_Level": "LOW",
        "Fabric_Mitigation": "Scheduled Fabric Data Factory pipeline using tumbling window trigger."
    },
    {
        "ID": "1.3",
        "Category": "Generation",
        "Question": "Stateful vs CDC Mechanism",
        "Status": "In Progress",
        "Answer_Details": "Evaluating CDC via Azure SQL Mirroring vs daily full snapshot replacement.",
        "Risk_Level": "MEDIUM",
        "Fabric_Mitigation": "Adopt Fabric Database Mirroring for zero-code CDC synchronization into OneLake."
    },
    {
        "ID": "1.4",
        "Category": "Generation",
        "Question": "Data Provider & Transmission Protocol",
        "Status": "Verified",
        "Answer_Details": "Internal database connected via On-Premises Data Gateway.",
        "Risk_Level": "LOW",
        "Fabric_Mitigation": "Data Factory Copy Activity using existing Gateway cluster."
    },
    {
        "ID": "1.5",
        "Category": "Generation",
        "Question": "Source System Read Performance Impact",
        "Status": "Verified",
        "Answer_Details": "Dedicated read-replica provisioned for extraction; no production lock impact.",
        "Risk_Level": "LOW",
        "Fabric_Mitigation": "Connect pipeline directly to read-only replica connection string."
    },
    {
        "ID": "1.6",
        "Category": "Generation",
        "Question": "Upstream Dependency Characteristics",
        "Status": "Action Required",
        "Answer_Details": "Source DB relies on external vendor feed at 00:00 UTC which occasionally delays.",
        "Risk_Level": "MEDIUM",
        "Fabric_Mitigation": "Add pre-execution validation activity in Fabric Pipeline checking file watermarks."
    },
    {
        "ID": "1.7",
        "Category": "Generation",
        "Question": "Late / Missing Data Quality Checks",
        "Status": "In Progress",
        "Answer_Details": "Basic row count validation planned in Spark notebook.",
        "Risk_Level": "MEDIUM",
        "Fabric_Mitigation": "Use Great Expectations / PySpark expectations to log validation telemetry to Lakehouse."
    }
]

df_source = pd.DataFrame(source_assessment)
display(df_source)


---
## 2. Storage Systems (Book Reference: Page 38)
> *"Evaluating storage requires looking at write/read speeds, bottlenecks, and future scale."* — Reis & Housley

### 📋 Architectural Questions & Microsoft Fabric Context

1. **Read/Write Speed Compatibility:** *Is this storage solution compatible with the architecture’s required write and read speeds?*
   * **Fabric Context:** In OneLake, Delta Lake parquet format provides high-throughput sequential reads. Enable **V-Order** optimization on all writes to accelerate query and Power BI Direct Lake read speeds.
2. **Downstream Bottlenecks:** *Will storage create a bottleneck for downstream processes?*
   * **Fabric Context:** Watch out for the "small file problem" from micro-batching. Run scheduled `OPTIMIZE` (compaction) and `VACUUM` jobs on Delta tables.
3. **Storage Mechanics & "Unnatural Acts":** *Do you understand how this storage technology works? Are you utilizing the storage system optimally or committing "unnatural acts" (like applying a high rate of random access updates in an object storage system)?*
   * **Fabric Context:** Parquet files in OneLake are immutable. Performing thousands of row-by-row OLTP-style updates causes severe write amplification and capacity burnout. Use batch `MERGE` operations or append-only event logs.
4. **Anticipated Future Scale:** *Will this storage system handle anticipated future scale? (Consider total available storage, read operation rate, write volume, etc.)*
   * **Fabric Context:** OneLake storage scales automatically. Monitor your **Fabric Capacity Metrics App** to ensure compute capacity (CU) can support peak compaction, ingestion, and query concurrency.
5. **SLA Retrieval Speed:** *Will downstream users and processes be able to retrieve data within the required service-level agreement (SLA)?*
   * **Fabric Context:** Architect Gold models specifically for **Power BI Direct Lake mode**, which reads parquet files directly from OneLake without caching or translation.
6. **Metadata & Lineage:** *Are you capturing metadata about schema evolution, data flows, and data lineage?*
   * **Fabric Context:** Fabric natively captures lineage across Lakehouses, Notebooks, Pipelines, and Power BI. Connect to **Microsoft Purview** for comprehensive enterprise metadata search.
7. **Pure Storage vs Complex Queries:** *Is this a pure storage solution (object storage) or does it support complex query patterns (like a cloud data warehouse)?*
   * **Fabric Context:** Understand the roles: **Fabric Lakehouse** provides file + Spark engine flexibility; **Fabric Data Warehouse** provides ACID T-SQL relational engine with cross-database querying over OneLake.
8. **Schema Rigidity:** *Is the storage system schema-agnostic, flexible schema, or enforced schema?*
   * **Fabric Context:** Implement the **Medallion Architecture**: Bronze = Schema-flexible/Agnostic; Silver = Conformed & Cleaned; Gold = Strict Schema Enforcement & Business Rules.
9. **Governance & Golden Records:** *How are you tracking master data, golden records, data quality, and data lineage for data governance?*
   * **Fabric Context:** Build a centralized Gold Semantic Model in OneLake; certify datasets in Fabric workspace; designate official golden records through Microsoft Purview Master Data Management.
10. **Regulatory Compliance & Data Sovereignty:** *How are you handling regulatory compliance and data sovereignty (e.g., storing data in specific geographical locations)?*
    * **Fabric Context:** Deploy Fabric Capacities in specific Azure regions; use **OneLake Shortcuts** to reference data in external clouds (AWS S3, ADLS Gen2, Google Cloud Storage) without copying it across boundaries.


In [ ]:
# ==============================================================================
# 🛠️ STAGE 2: STORAGE SYSTEM DISCOVERY MATRIX
# ==============================================================================
storage_assessment = [
    {
        "ID": "2.1",
        "Category": "Storage",
        "Question": "Read/Write Speed Compatibility",
        "Status": "Verified",
        "Answer_Details": "OneLake Delta Parquet with V-Order meets read/write requirements.",
        "Risk_Level": "LOW",
        "Fabric_Mitigation": "Ensure V-Order is enabled on all Silver and Gold Delta tables."
    },
    {
        "ID": "2.2",
        "Category": "Storage",
        "Question": "Downstream Bottlenecks & Small Files",
        "Status": "Action Required",
        "Answer_Details": "Streaming ingestion generates thousands of small files per hour.",
        "Risk_Level": "HIGH",
        "Fabric_Mitigation": "Configure scheduled maintenance job running OPTIMIZE and VACUUM every 6 hours."
    },
    {
        "ID": "2.3",
        "Category": "Storage",
        "Question": "Technology Mechanics & Avoiding Unnatural Acts",
        "Status": "Verified",
        "Answer_Details": "No random row-level updates; operations designed as append-only Bronze and batch merge Silver.",
        "Risk_Level": "LOW",
        "Fabric_Mitigation": "Enforce micro-batch MERGE patterns in PySpark notebooks."
    },
    {
        "ID": "2.4",
        "Category": "Storage",
        "Question": "Anticipated Future Scale (Storage & Compute CU)",
        "Status": "In Progress",
        "Answer_Details": "Dataset growing from 500GB to 10TB over next 18 months.",
        "Risk_Level": "MEDIUM",
        "Fabric_Mitigation": "Monitor Fabric Capacity Metrics app and evaluate upgrading capacity as scale dictates."
    },
    {
        "ID": "2.5",
        "Category": "Storage",
        "Question": "Downstream SLA & Power BI Retrieval Speed",
        "Status": "Verified",
        "Answer_Details": "Gold reporting layer designed specifically for Power BI Direct Lake mode.",
        "Risk_Level": "LOW",
        "Fabric_Mitigation": "Validate that semantic model stays strictly within Direct Lake without falling back to DirectQuery."
    },
    {
        "ID": "2.6",
        "Category": "Storage",
        "Question": "Metadata, Schema Evolution & Lineage Capture",
        "Status": "Verified",
        "Answer_Details": "Native Fabric Lineage view active; Delta Lake transaction logs track snapshot history.",
        "Risk_Level": "LOW",
        "Fabric_Mitigation": "Connect workspace to Microsoft Purview catalog."
    },
    {
        "ID": "2.7",
        "Category": "Storage",
        "Question": "Pure Storage vs Complex Relational Query Support",
        "Status": "Verified",
        "Answer_Details": "Lakehouse selected for engineering; SQL Analytics Endpoint used for analyst ad-hoc queries.",
        "Risk_Level": "LOW",
        "Fabric_Mitigation": "Utilize SQL Endpoint automatically generated for each Lakehouse."
    },
    {
        "ID": "2.8",
        "Category": "Storage",
        "Question": "Schema Rigidity Strategy Across Medallion Layers",
        "Status": "Verified",
        "Answer_Details": "Bronze is permissive; Silver enforces types; Gold enforces relational constraints.",
        "Risk_Level": "LOW",
        "Fabric_Mitigation": "Use explicit StructType schemas on Silver/Gold writes."
    },
    {
        "ID": "2.9",
        "Category": "Storage",
        "Question": "Master Data, Golden Records & Governance",
        "Status": "Action Required",
        "Answer_Details": "Customer dimension duplicated across 3 departments with conflicting IDs.",
        "Risk_Level": "HIGH",
        "Fabric_Mitigation": "Establish a centralized Customer Master Data model in the Enterprise Gold Lakehouse."
    },
    {
        "ID": "2.10",
        "Category": "Storage",
        "Question": "Regulatory Compliance & Data Sovereignty",
        "Status": "Verified",
        "Answer_Details": "All data resides in Azure East US 2 capacity; compliant with domestic privacy policies.",
        "Risk_Level": "LOW",
        "Fabric_Mitigation": "Lock workspace assignment to verified regional capacity."
    }
]

df_storage = pd.DataFrame(storage_assessment)
display(df_storage)


---
## 3. Ingestion Phase (Book Reference: Page 40)
> *"The ingestion stage is often a major bottleneck, making these considerations crucial."* — Reis & Housley

### 📋 Architectural Questions & Microsoft Fabric Context

1. **Use Cases & Reusability:** *What are the use cases for the data I’m ingesting? Can I reuse this data rather than create multiple versions of the same dataset?*
   * **Fabric Context:** Utilize **OneLake Shortcuts** ("One Copy" principle) to reuse ingested Bronze/Silver tables across multiple workspaces without data duplication.
2. **Generation & Ingestion Reliability:** *Are the systems generating and ingesting this data reliably, and is the data available when I need it?*
   * **Fabric Context:** Configure automated retry policies and alerts in Fabric Pipelines. Inspect pipeline runs via the Fabric Monitoring Hub.
3. **Destination After Ingestion:** *What is the data destination after ingestion?*
   * **Fabric Context:** Standardize on **Medallion Landing**: Raw files land in Lakehouse `Files/bronze_landing/`; parsed tables land in Lakehouse `Tables/bronze_delta/`.
4. **Access Frequency Profile:** *How frequently will I need to access the data?*
   * **Fabric Context:** Hot tables remain in active Delta tables; cold raw archives are stored in compressed parquet partitions.
5. **Arrival Volume Profile:** *In what volume will the data typically arrive?*
   * **Fabric Context:** For small frequent payloads (KB/MB), use Fabric Eventstreams or Dataflow Gen2. For large bulk uploads (GB/TB), use Pipeline Copy Activity with high-throughput staging.
6. **Format Compatibility:** *What format is the data in? Can my downstream storage and transformation systems handle this format?*
   * **Fabric Context:** Immediately convert semi-structured formats (CSV, XML, JSON) into Delta Parquet upon landing in the Lakehouse.
7. **Source Data Usability:** *Is the source data in good shape for immediate downstream use? If so, for how long, and what may cause it to be unusable?*
   * **Fabric Context:** Never point reporting tools directly to raw ingested data. Always enforce a conformed **Silver layer** for validation.
8. **In-Flight Transformations:** *If the data is from a streaming source, does it need to be transformed before reaching its destination (e.g., in-flight transformations)?*
   * **Fabric Context:** Use **Fabric Eventstreams** for light in-flight routing, filtering, and tumbling aggregations; use Spark Structured Streaming for complex stateful joins and watermarking.


In [ ]:
# ==============================================================================
# 🛠️ STAGE 3: INGESTION PHASE DISCOVERY MATRIX
# ==============================================================================
ingestion_assessment = [
    {
        "ID": "3.1",
        "Category": "Ingestion",
        "Question": "Use Case Clarity & Data Reusability",
        "Status": "Verified",
        "Answer_Details": "Single enterprise ingestion of transactional sales; shared with Finance and Marketing via OneLake Shortcuts.",
        "Risk_Level": "LOW",
        "Fabric_Mitigation": "Provision OneLake shortcuts in departmental workspaces pointing to central Lakehouse."
    },
    {
        "ID": "3.2",
        "Category": "Ingestion",
        "Question": "Generation & Ingestion Reliability",
        "Status": "In Progress",
        "Answer_Details": "Source API occasionally throttles requests with HTTP 429.",
        "Risk_Level": "MEDIUM",
        "Fabric_Mitigation": "Configure exponential backoff retry policy in Data Factory Web Activity."
    },
    {
        "ID": "3.3",
        "Category": "Ingestion",
        "Question": "Clear Ingestion Destination",
        "Status": "Verified",
        "Answer_Details": "Landing in Bronze Lakehouse Files folder, followed immediately by PySpark ingestion into Delta table.",
        "Risk_Level": "LOW",
        "Fabric_Mitigation": "Standardized folder structure: Lakehouse/Files/landing/{source_name}/{yyyy}/{mm}/{dd}/"
    },
    {
        "ID": "3.4",
        "Category": "Ingestion",
        "Question": "Access Frequency Profile",
        "Status": "Verified",
        "Answer_Details": "Queried continuously throughout business day by Power BI reports and ad-hoc analysts.",
        "Risk_Level": "LOW",
        "Fabric_Mitigation": "Maintain in high-performance Delta tables with V-Order enabled."
    },
    {
        "ID": "3.5",
        "Category": "Ingestion",
        "Question": "Expected Volume & Surge Capacity",
        "Status": "In Progress",
        "Answer_Details": "Standard volume is 2GB/day; end-of-month surges reach 50GB.",
        "Risk_Level": "MEDIUM",
        "Fabric_Mitigation": "Ensure Spark pool autoscaling is configured to scale up worker nodes during surge window."
    },
    {
        "ID": "3.6",
        "Category": "Ingestion",
        "Question": "Format Compatibility & Conversion",
        "Status": "Verified",
        "Answer_Details": "Source delivers multi-line JSON; PySpark cleans and writes to Delta Parquet.",
        "Risk_Level": "LOW",
        "Fabric_Mitigation": "Use spark.read.json() with schema hints."
    },
    {
        "ID": "3.7",
        "Category": "Ingestion",
        "Question": "Source Data Shelf-Life & Usability",
        "Status": "Verified",
        "Answer_Details": "Raw data requires currency conversion and timezone normalization before usable.",
        "Risk_Level": "LOW",
        "Fabric_Mitigation": "Isolate all business conversions strictly inside Silver transformation layer."
    },
    {
        "ID": "3.8",
        "Category": "Ingestion",
        "Question": "In-Flight Transformation Requirements",
        "Status": "Not Applicable",
        "Answer_Details": "Batch workload only; no real-time streaming requirements identified for Phase 1.",
        "Risk_Level": "LOW",
        "Fabric_Mitigation": "N/A - Standard scheduled batch pipeline."
    }
]

df_ingestion = pd.DataFrame(ingestion_assessment)
display(df_ingestion)


---
## 4. Transformation Phase (Book Reference: Page 43)
> *"Transformations are where data begins to create tangible value, which requires asking these fundamental questions."* — Reis & Housley

### 📋 Architectural Questions & Microsoft Fabric Context

1. **Cost, ROI & Business Value:** *What’s the cost and return on investment (ROI) of the transformation? What is the associated business value?*
   * **Fabric Context:** Measure the compute cost in Fabric Capacity Units (CU) via the **Capacity Metrics App**. Avoid running hourly Spark jobs for dashboards viewed only once a week.
2. **Simplicity & Isolation:** *Is the transformation as simple and self-isolated as possible?*
   * **Fabric Context:** Build modular, idempotent transformations. Separate cleaning (Silver) from business modeling (Gold). Use modular Fabric Notebooks or **dbt (data build tool)** for version-controlled, testable transformations.
3. **Business Rules Support:** *What business rules do the transformations support?*
   * **Fabric Context:** Document business calculation logic in a data dictionary. Decide whether logic belongs in the Gold storage layer (pre-computed dimensions/facts) or in the Power BI Semantic Model (dynamic DAX measures).


In [ ]:
# ==============================================================================
# 🛠️ STAGE 4: TRANSFORMATION PHASE DISCOVERY MATRIX
# ==============================================================================
transformation_assessment = [
    {
        "ID": "4.1",
        "Category": "Transformation",
        "Question": "Cost, ROI & Associated Business Value",
        "Status": "In Progress",
        "Answer_Details": "Pipeline powers executive revenue KPI dashboard; estimated business value is $250k/year in decision agility.",
        "Risk_Level": "LOW",
        "Fabric_Mitigation": "Track CU consumption per notebook run; keep total Fabric compute cost under $500/month."
    },
    {
        "ID": "4.2",
        "Category": "Transformation",
        "Question": "Simplicity, Isolation & Idempotence",
        "Status": "Action Required",
        "Answer_Details": "Existing script is a 1,500-line legacy stored procedure doing 15 operations in one step.",
        "Risk_Level": "CRITICAL",
        "Fabric_Mitigation": "Refactor into discrete PySpark notebooks: 01_Bronze_Cleanse -> 02_Silver_Conform -> 03_Gold_Aggregate with idempotent MERGE statements."
    },
    {
        "ID": "4.3",
        "Category": "Transformation",
        "Question": "Business Rules & Calculation Ownership",
        "Status": "In Progress",
        "Answer_Details": "Dispute between Sales and Finance on definition of 'Active Customer' and 'Net Revenue'.",
        "Risk_Level": "HIGH",
        "Fabric_Mitigation": "Align definitions with business leads before coding; codify agreed rules into conformed Gold semantic model."
    }
]

df_transformation = pd.DataFrame(transformation_assessment)
display(df_transformation)


---
## 5. Serving Data (Specific to Machine Learning) (Book Reference: Page 46)
> *"While serving data covers analytics, ML, and reverse ETL, the authors highlight these considerations specifically for ML deployments."* — Reis & Housley

### 📋 Architectural Questions & Microsoft Fabric Context

1. **Feature Engineering Data Quality:** *Is the data of sufficient quality to perform reliable feature engineering?*
   * **Fabric Context:** Check for null distributions, class skews, and temporal leakage. Use Delta time-travel to reproduce exact historical feature training sets.
2. **Discoverability:** *Is the data discoverable? Can data scientists and ML engineers easily find valuable data?*
   * **Fabric Context:** Register curated feature datasets in the **Fabric OneLake Data Hub**. Tag and annotate tables with business descriptions and endorsements (Promoted/Certified).
3. **Technical & Organizational Boundaries:** *Where are the technical and organizational boundaries between data engineering and ML engineering?*
   * **Fabric Context:** Define clear ownership: **Data Engineers** build and SLA-guarantee reliable Bronze/Silver/Gold pipelines up to conformed Feature Store tables in OneLake. **ML Engineers** own feature engineering, model training, **MLflow experiment tracking**, and inference endpoints in Fabric Synapse Data Science.
4. **Ground Truth & Bias Representation:** *Does the dataset properly represent ground truth? Is it unfairly biased?*
   * **Fabric Context:** Conduct bias audits; inspect class distributions, geographic skews, and historical bias. Leverage open-source responsible AI frameworks (Fairlearn) directly in Fabric PySpark notebooks.


In [ ]:
# ==============================================================================
# 🛠️ STAGE 5: SERVING DATA (ML FOCUS) DISCOVERY MATRIX
# ==============================================================================
serving_ml_assessment = [
    {
        "ID": "5.1",
        "Category": "Serving (ML)",
        "Question": "Data Quality for Feature Engineering",
        "Status": "Action Required",
        "Answer_Details": "Customer churn labels contain 12% missing historical outcome records.",
        "Risk_Level": "HIGH",
        "Fabric_Mitigation": "Cleanse and impute ground truth labels in Silver layer; drop indeterminate records before feature generation."
    },
    {
        "ID": "5.2",
        "Category": "Serving (ML)",
        "Question": "Discoverability & Feature Cataloging",
        "Status": "Verified",
        "Answer_Details": "Gold Lakehouse feature tables registered in OneLake Data Hub with data dictionary.",
        "Risk_Level": "LOW",
        "Fabric_Mitigation": "Certify the Lakehouse in the Fabric workspace to promote visibility across data science teams."
    },
    {
        "ID": "5.3",
        "Category": "Serving (ML)",
        "Question": "DE vs MLE Organizational & Technical Boundary",
        "Status": "In Progress",
        "Answer_Details": "Team is clarifying who maintains scheduled batch model scoring pipelines.",
        "Risk_Level": "MEDIUM",
        "Fabric_Mitigation": "DE team schedules Fabric Pipeline executing MLE-authored PySpark scoring notebook with MLflow."
    },
    {
        "ID": "5.4",
        "Category": "Serving (ML)",
        "Question": "Ground Truth Representation & Bias Audit",
        "Status": "Action Required",
        "Answer_Details": "Training data is heavily skewed toward top 5 metropolitan markets; rural customers underrepresented.",
        "Risk_Level": "HIGH",
        "Fabric_Mitigation": "Perform demographic stratification and apply re-weighting or Fairlearn audit in Spark training notebook."
    }
]

df_serving_ml = pd.DataFrame(serving_ml_assessment)
display(df_serving_ml)


---
## 6. Executive Readiness Scorecard & Summary Dashboard
This cell aggregates your assessment across all 5 lifecycle phases, calculates a **Data Architecture Readiness Score (0-100%)**, highlights unmitigated critical risks, and produces an executive summary table.


In [ ]:
# ==============================================================================
# 📊 CONSOLIDATED READINESS SCORECARD & AUDIT GENERATOR
# ==============================================================================
import pandas as pd
import numpy as np

all_assessments = (
    source_assessment + 
    storage_assessment + 
    ingestion_assessment + 
    transformation_assessment + 
    serving_ml_assessment
)
df_full = pd.DataFrame(all_assessments)

status_map = {"Verified": 100, "In Progress": 50, "Action Required": 0, "Not Applicable": 100}
df_full["Status_Score"] = df_full["Status"].map(status_map)

total_questions = len(df_full)
verified_count = (df_full["Status"] == "Verified").sum()
in_progress_count = (df_full["Status"] == "In Progress").sum()
action_required_count = (df_full["Status"] == "Action Required").sum()
critical_risks = (df_full["Risk_Level"] == "CRITICAL").sum()
high_risks = (df_full["Risk_Level"] == "HIGH").sum()

overall_score = round(df_full["Status_Score"].mean(), 1)

print("=" * 80)
print(f"📊 MICROSOFT FABRIC DATA PLAN READINESS REPORT")
print("=" * 80)
print(f"Overall Architectural Readiness Score: {overall_score} / 100")
print(f"Total Questions Assessed:             {total_questions}")
print(f"  • Verified & Ready:                  {verified_count}")
print(f"  • In Progress:                       {in_progress_count}")
print(f"  • Action Required (Blocking):        {action_required_count}")
print(f"Risk Profile:")
print(f"  • Critical Risks:                    {critical_risks}")
print(f"  • High Risks:                        {high_risks}")
print("=" * 80)

if overall_score >= 85 and critical_risks == 0:
    print("🟢 STATUS: GREEN (Ready for Implementation)")
    print("   The architectural plan is robust. Proceed with pipeline deployment.")
elif overall_score >= 65 and critical_risks == 0:
    print("🟡 STATUS: YELLOW (Conditional Approval)")
    print("   Address Action Required items before promoting pipelines to production.")
else:
    print("🔴 STATUS: RED (Implementation Blocked)")
    print("   Critical architectural gaps exist. Resolve High/Critical risks before coding.")

print("=" * 80)

print("\n📈 READINESS SCORE BY LIFECYCLE STAGE:")
df_cat = df_full.groupby("Category").agg(
    Questions=("ID", "count"),
    Avg_Readiness=("Status_Score", "mean"),
    Action_Required=("Status", lambda s: (s == "Action Required").sum()),
    High_Or_Critical_Risks=("Risk_Level", lambda r: r.isin(["HIGH", "CRITICAL"]).sum())
).round(1).reset_index()

display(df_cat)

print("\n🚨 HIGH-PRIORITY ACTION ITEMS (Must resolve before production):")
df_urgent = df_full[df_full["Risk_Level"].isin(["CRITICAL", "HIGH"]) | (df_full["Status"] == "Action Required")][
    ["ID", "Category", "Question", "Status", "Risk_Level", "Fabric_Mitigation"]
]
display(df_urgent)


### 💾 (Optional) Persist Assessment to Fabric OneLake Delta Table
When running inside Microsoft Fabric Synapse Data Engineering, execute the following cell to record this audit snapshot into your governance catalog:


In [ ]:
# ==============================================================================
# 💾 OPTIONAL: PERSIST AUDIT SNAPSHOT TO ONELAKE DELTA TABLE
# (Uncomment when running in Microsoft Fabric Synapse Spark environment)
# ==============================================================================
# from pyspark.sql import SparkSession
# from datetime import datetime

# spark = SparkSession.builder.getOrCreate()
# df_full["Assessment_Timestamp"] = datetime.utcnow()
# df_full["Assessed_By"] = spark.conf.get("spark.ms.autogen.cluster.username", "fabric_architect")

# spark_df = spark.createDataFrame(df_full.drop(columns=["Status_Score"]))
# spark_df.write \
#     .format("delta") \
#     .mode("append") \
#     .option("mergeSchema", "true") \
#     .saveAsTable("lakehouse.data_governance.project_readiness_audits")

# print("✅ Successfully logged assessment to data_governance.project_readiness_audits Delta table.")
